## **X03 STEP 1 — OFFICIAL INPUT INVENTORY AND PAIRED XAI VERIFICATION**

The official X01 and X02 schemas confirm that the frozen publication subset is stored through the `publication_subset` marker within the shared registry and both XAI metadata tables.

This step now performs the <u>final paired verification</u> of the **20 frozen sample IDs**, four predefined groups, *F06 true/predicted labels*, predicted-class explanation targets, publication-subset membership, and locked F06 checkpoint identity.

*No Grad-CAM–LIME spatial comparison is performed in this step.*

In [17]:
from pathlib import Path
import pandas as pd

print("=" * 80)
print("X03 STEP 1 — OFFICIAL INPUT INVENTORY AND PAIRED XAI VERIFICATION")
print("=" * 80)

X01 = Path("/kaggle/input/datasets/sazzad1/aqua20-x01-f06-final-gradcam-outputs")
X02 = Path("/kaggle/input/datasets/sazzad1/aqua20-x02-f06-final-lime-outputs")

LOCKED_SHA = "d0c225c28b25f09822bc99518a37f9b1fe642b2d01213da7f68e2d76a255832a"

r1 = pd.read_csv(X01 / "tables/X01_frozen_xai_sample_registry.csv")
r2 = pd.read_csv(X02 / "tables/X01_frozen_xai_sample_registry.csv")
g = pd.read_csv(X01 / "tables/X01_gradcam_metadata.csv")
l = pd.read_csv(X02 / "tables/X02_lime_metadata.csv")

for df in [r1, r2, g, l]:
    df.sort_values("frozen_sample_order", inplace=True)
    df.reset_index(drop=True, inplace=True)

pair_cols = [
    "sample_id",
    "xai_group",
    "true_class_id",
    "true_class_name",
    "predicted_class_id",
    "predicted_class_name",
    "explanation_target_class_id",
    "explanation_target_class_name",
]

registry_match = r1.equals(r2)
sample_ids_match = r1["sample_id"].equals(r2["sample_id"])
labels_match = r1[pair_cols].equals(r2[pair_cols])
metadata_match = g[pair_cols].equals(l[pair_cols])

registry_meta_match = (
    r1[pair_cols].equals(g[pair_cols])
    and r1[pair_cols].equals(l[pair_cols])
)

target_is_prediction = (
    (r1["explanation_target_class_id"] == r1["predicted_class_id"]).all()
    and (g["explanation_target_class_id"] == g["predicted_class_id"]).all()
    and (l["explanation_target_class_id"] == l["predicted_class_id"]).all()
)

pub_r1 = r1.loc[r1["publication_subset"], "sample_id"].tolist()
pub_r2 = r2.loc[r2["publication_subset"], "sample_id"].tolist()
pub_g = g.loc[g["publication_subset"], "sample_id"].tolist()
pub_l = l.loc[l["publication_subset"], "sample_id"].tolist()

publication_match = pub_r1 == pub_r2 == pub_g == pub_l

group_counts = r1["xai_group"].value_counts().sort_index()
group_structure_ok = (
    len(r1) == 20
    and r1["sample_id"].is_unique
    and len(group_counts) == 4
    and (group_counts == 5).all()
)

checkpoint_match = (
    g["checkpoint_sha256"].nunique() == 1
    and l["checkpoint_sha256"].nunique() == 1
    and g["checkpoint_sha256"].iloc[0] == LOCKED_SHA
    and l["checkpoint_sha256"].iloc[0] == LOCKED_SHA
)

print(f"X01 registry rows                : {len(r1)}")
print(f"X02 registry rows                : {len(r2)}")
print(f"X01 Grad-CAM metadata rows       : {len(g)}")
print(f"X02 LIME metadata rows           : {len(l)}")
print(f"Publication subset size          : {len(pub_r1)}")

print("\nFrozen group counts:")
print(group_counts.to_string())

print("\n" + "-" * 80)
print(f"Shared frozen registry identical : {'PASS' if registry_match else 'FAIL'}")
print(f"One-to-one sample IDs            : {'PASS' if sample_ids_match else 'FAIL'}")
print(f"Four groups × five samples       : {'PASS' if group_structure_ok else 'FAIL'}")
print(f"Identical F06 true/pred labels   : {'PASS' if labels_match else 'FAIL'}")
print(f"X01/X02 metadata labels paired   : {'PASS' if metadata_match else 'FAIL'}")
print(f"Registry ↔ metadata correspondence: {'PASS' if registry_meta_match else 'FAIL'}")
print(f"Predicted class explained        : {'PASS' if target_is_prediction else 'FAIL'}")
print(f"Publication subset unchanged     : {'PASS' if publication_match else 'FAIL'}")
print(f"Locked F06 checkpoint verified   : {'PASS' if checkpoint_match else 'FAIL'}")

ready = all([
    registry_match,
    sample_ids_match,
    group_structure_ok,
    labels_match,
    metadata_match,
    registry_meta_match,
    target_is_prediction,
    publication_match,
    checkpoint_match,
])

print("-" * 80)
print(f"X03 READY TO PROCEED             : {'YES' if ready else 'NO'}")
print("=" * 80)

assert ready, "X03 Step 1 verification failed."

X03 STEP 1 — OFFICIAL INPUT INVENTORY AND PAIRED XAI VERIFICATION
X01 registry rows                : 20
X02 registry rows                : 20
X01 Grad-CAM metadata rows       : 20
X02 LIME metadata rows           : 20
Publication subset size          : 12

Frozen group counts:
xai_group
correct_difficult                5
correct_high_confidence          5
minority_weak_class              5
misclassified_high_confidence    5

--------------------------------------------------------------------------------
Shared frozen registry identical : PASS
One-to-one sample IDs            : PASS
Four groups × five samples       : PASS
Identical F06 true/pred labels   : PASS
X01/X02 metadata labels paired   : PASS
Registry ↔ metadata correspondence: PASS
Predicted class explained        : PASS
Publication subset unchanged     : PASS
Locked F06 checkpoint verified   : PASS
--------------------------------------------------------------------------------
X03 READY TO PROCEED             : YES


## **X03 STEP 2 — PAIRED EXPLANATION ARTIFACT AND SPATIAL REPRESENTATION VERIFICATION**

This step verifies the paired **Grad-CAM heatmap** and **LIME positive-region mask** artifacts for every frozen XAI sample.

For each shared `sample_id`, the required explanation files are checked and their spatial dimensions, image modes, and stored value structures are inspected to confirm that the two methods can be compared on the <u>same image coordinate space</u>.

*No Grad-CAM–LIME overlap, convergence, or disagreement metric is calculated in this step.*

In [18]:
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np

print("=" * 80)
print("X03 STEP 2 — PAIRED EXPLANATION ARTIFACT AND SPATIAL REPRESENTATION VERIFICATION")
print("=" * 80)

X01 = Path("/kaggle/input/datasets/sazzad1/aqua20-x01-f06-final-gradcam-outputs")
X02 = Path("/kaggle/input/datasets/sazzad1/aqua20-x02-f06-final-lime-outputs")

g = pd.read_csv(X01 / "tables/X01_gradcam_metadata.csv")
l = pd.read_csv(X02 / "tables/X02_lime_metadata.csv")

paired = g[
    ["sample_id", "frozen_sample_order", "xai_group", "heatmap_file"]
].merge(
    l[["sample_id", "mask_file"]],
    on="sample_id",
    validate="one_to_one",
)

records = []

for row in paired.itertuples(index=False):
    grad_path = X01 / row.heatmap_file
    lime_path = X02 / row.mask_file

    assert grad_path.is_file(), f"Missing Grad-CAM heatmap: {grad_path}"
    assert lime_path.is_file(), f"Missing LIME mask: {lime_path}"

    grad_img = Image.open(grad_path)
    lime_img = Image.open(lime_path)

    grad = np.asarray(grad_img)
    lime = np.asarray(lime_img)

    records.append({
        "sample_id": row.sample_id,
        "xai_group": row.xai_group,
        "grad_size": grad_img.size,
        "lime_size": lime_img.size,
        "grad_mode": grad_img.mode,
        "lime_mode": lime_img.mode,
        "grad_min": grad.min(),
        "grad_max": grad.max(),
        "lime_unique_values": len(np.unique(lime)),
        "same_spatial_size": grad_img.size == lime_img.size,
    })

check = pd.DataFrame(records).sort_values("sample_id").reset_index(drop=True)

print(f"Paired samples                    : {len(check)}")
print(f"Grad-CAM heatmaps found           : {len(check)}")
print(f"LIME masks found                  : {len(check)}")
print(f"Same spatial dimensions           : {check['same_spatial_size'].all()}")

print("\nGrad-CAM sizes:")
print(check["grad_size"].value_counts().to_string())

print("\nLIME mask sizes:")
print(check["lime_size"].value_counts().to_string())

print("\nImage modes:")
print(check[["grad_mode", "lime_mode"]].value_counts().to_string())

print("\nStored value structure:")
print(f"Grad-CAM value range              : {check['grad_min'].min()} to {check['grad_max'].max()}")
print(f"LIME unique-value counts          : {sorted(check['lime_unique_values'].unique())}")

ready = (
    len(check) == 20
    and check["same_spatial_size"].all()
)

print("-" * 80)
print(f"PAIRED SPATIAL ARTIFACTS VERIFIED : {'PASS' if ready else 'FAIL'}")
print(f"X03 STEP 2 READY                  : {'YES' if ready else 'NO'}")
print("=" * 80)

assert ready, "X03 Step 2 artifact verification failed."

X03 STEP 2 — PAIRED EXPLANATION ARTIFACT AND SPATIAL REPRESENTATION VERIFICATION
Paired samples                    : 20
Grad-CAM heatmaps found           : 20
LIME masks found                  : 20
Same spatial dimensions           : True

Grad-CAM sizes:
grad_size
(256, 256)     16
(320, 240)      3
(1514, 912)     1

LIME mask sizes:
lime_size
(256, 256)     16
(320, 240)      3
(1514, 912)     1

Image modes:
grad_mode  lime_mode
RGB        L            20

Stored value structure:
Grad-CAM value range              : 0 to 255
LIME unique-value counts          : [np.int64(2)]
--------------------------------------------------------------------------------
PAIRED SPATIAL ARTIFACTS VERIFIED : PASS
X03 STEP 2 READY                  : YES


## **X03 STEP 3 — GRAD-CAM HEATMAP ENCODING AND SCALAR SALIENCY RECOVERY VERIFICATION**

The stored Grad-CAM heatmaps are **RGB images**, whereas LIME provides binary positive-region masks. Before direct comparison, the Grad-CAM heatmap encoding must be verified so that a defensible *scalar activation map* can be recovered from the saved artifact.

This step tests whether the Grad-CAM heatmaps use the expected **OpenCV JET color-map encoding** and whether their pixels can be mapped back to activation levels on the original spatial grid.

<u>No Grad-CAM–LIME convergence or overlap metric is calculated in this step.</u>

In [19]:
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import cv2

print("=" * 80)
print("X03 STEP 3 — GRAD-CAM HEATMAP ENCODING AND SCALAR SALIENCY RECOVERY VERIFICATION")
print("=" * 80)

X01 = Path("/kaggle/input/datasets/sazzad1/aqua20-x01-f06-final-gradcam-outputs")
g = pd.read_csv(X01 / "tables/X01_gradcam_metadata.csv")

jet_bgr = cv2.applyColorMap(
    np.arange(256, dtype=np.uint8).reshape(256, 1),
    cv2.COLORMAP_JET
).reshape(256, 3)

jet_rgb = jet_bgr[:, ::-1]
jet_lookup = {tuple(rgb): i for i, rgb in enumerate(jet_rgb)}

records = []

for row in g.itertuples(index=False):
    heat = np.asarray(Image.open(X01 / row.heatmap_file).convert("RGB"))

    flat = heat.reshape(-1, 3)
    unique_rgb = np.unique(flat, axis=0)

    matched = np.array([
        tuple(rgb) in jet_lookup
        for rgb in flat
    ])

    recovered = np.array([
        jet_lookup.get(tuple(rgb), -1)
        for rgb in flat
    ])

    records.append({
        "sample_id": row.sample_id,
        "xai_group": row.xai_group,
        "grayscale_rgb": np.array_equal(heat[..., 0], heat[..., 1])
                         and np.array_equal(heat[..., 1], heat[..., 2]),
        "unique_rgb_colors": len(unique_rgb),
        "jet_match_fraction": matched.mean(),
        "recovered_min": recovered[recovered >= 0].min(),
        "recovered_max": recovered[recovered >= 0].max(),
    })

encoding = pd.DataFrame(records)

print(f"Heatmaps inspected                : {len(encoding)}")
print(f"Grayscale-replicated heatmaps     : {encoding['grayscale_rgb'].sum()}")
print(f"Exact JET match — minimum fraction: {encoding['jet_match_fraction'].min():.6f}")
print(f"Exact JET match — mean fraction   : {encoding['jet_match_fraction'].mean():.6f}")
print(f"Exact JET match — maximum fraction: {encoding['jet_match_fraction'].max():.6f}")
print(f"Recovered activation range        : {encoding['recovered_min'].min()} to "
      f"{encoding['recovered_max'].max()}")

print("\nUnique RGB color counts:")
print(encoding["unique_rgb_colors"].value_counts().sort_index().to_string())

jet_verified = (
    len(encoding) == 20
    and (encoding["jet_match_fraction"] == 1.0).all()
)

print("-" * 80)
print(f"JET ENCODING VERIFIED             : {'PASS' if jet_verified else 'FAIL'}")
print(f"SCALAR SALIENCY RECOVERY READY    : {'YES' if jet_verified else 'NO'}")
print("=" * 80)

assert jet_verified, "Stored Grad-CAM heatmap encoding requires further inspection."

X03 STEP 3 — GRAD-CAM HEATMAP ENCODING AND SCALAR SALIENCY RECOVERY VERIFICATION
Heatmaps inspected                : 20
Grayscale-replicated heatmaps     : 0
Exact JET match — minimum fraction: 0.199142
Exact JET match — mean fraction   : 0.316180
Exact JET match — maximum fraction: 0.501617
Recovered activation range        : 1 to 253

Unique RGB color counts:
unique_rgb_colors
240    1
241    2
243    2
245    1
246    2
247    3
248    3
249    2
250    1
251    3
--------------------------------------------------------------------------------
JET ENCODING VERIFIED             : FAIL
SCALAR SALIENCY RECOVERY READY    : NO


AssertionError: Stored Grad-CAM heatmap encoding requires further inspection.

### **X03 STEP 3A — CORRECTED GRAD-CAM COLOR-DISTANCE INSPECTION**

The previous nearest-palette calculation suffered from *integer overflow during squared RGB-distance computation*, producing invalid `NaN` values.

This corrected inspection uses floating-point RGB arithmetic to quantify the relationship between the stored **lossless PNG Grad-CAM heatmaps** and the OpenCV JET palette.

<u>No Grad-CAM–LIME comparison or overlap metric is performed.</u>

In [21]:
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import cv2

print("=" * 80)
print("X03 STEP 3A — CORRECTED GRAD-CAM COLOR-DISTANCE INSPECTION")
print("=" * 80)

X01 = Path("/kaggle/input/datasets/sazzad1/aqua20-x01-f06-final-gradcam-outputs")
g = pd.read_csv(X01 / "tables/X01_gradcam_metadata.csv")

jet_rgb = cv2.applyColorMap(
    np.arange(256, dtype=np.uint8).reshape(256, 1),
    cv2.COLORMAP_JET
).reshape(256, 3)[:, ::-1].astype(np.float32)

records = []

for row in g.itertuples(index=False):
    path = X01 / row.heatmap_file

    with Image.open(path) as img:
        image_format = img.format
        heat = np.asarray(img.convert("RGB"))

    colors, counts = np.unique(
        heat.reshape(-1, 3),
        axis=0,
        return_counts=True
    )

    diff = colors[:, None, :].astype(np.float32) - jet_rgb[None, :, :]
    distances = np.sqrt(np.sum(diff ** 2, axis=2))

    nearest_distance = distances.min(axis=1)
    nearest_level = distances.argmin(axis=1)

    pixel_distances = np.repeat(nearest_distance, counts)

    records.append({
        "sample_id": row.sample_id,
        "image_format": image_format,
        "mean_nearest_rgb_distance": np.average(nearest_distance, weights=counts),
        "p95_nearest_rgb_distance": np.percentile(pixel_distances, 95),
        "max_nearest_rgb_distance": nearest_distance.max(),
        "recovered_min_level": nearest_level.min(),
        "recovered_max_level": nearest_level.max(),
    })

result = pd.DataFrame(records)

print(f"Heatmaps inspected                 : {len(result)}")
print(f"Stored image format                : {sorted(result['image_format'].unique())}")

print("\nNearest OpenCV JET palette distance:")
print(f"Mean pixel distance — minimum      : {result['mean_nearest_rgb_distance'].min():.4f}")
print(f"Mean pixel distance — mean         : {result['mean_nearest_rgb_distance'].mean():.4f}")
print(f"Mean pixel distance — maximum      : {result['mean_nearest_rgb_distance'].max():.4f}")
print(f"P95 pixel distance — maximum       : {result['p95_nearest_rgb_distance'].max():.4f}")
print(f"Maximum observed color distance    : {result['max_nearest_rgb_distance'].max():.4f}")

print("\nNearest-palette recovered levels:")
print(f"Recovered minimum                  : {result['recovered_min_level'].min()}")
print(f"Recovered maximum                  : {result['recovered_max_level'].max()}")

print("\nPer-sample summary:")
print(
    result[
        [
            "sample_id",
            "mean_nearest_rgb_distance",
            "p95_nearest_rgb_distance",
            "max_nearest_rgb_distance",
        ]
    ].to_string(index=False)
)

print("=" * 80)

X03 STEP 3A — CORRECTED GRAD-CAM COLOR-DISTANCE INSPECTION
Heatmaps inspected                 : 20
Stored image format                : ['PNG']

Nearest OpenCV JET palette distance:
Mean pixel distance — minimum      : 0.8052
Mean pixel distance — mean         : 1.9744
Mean pixel distance — maximum      : 4.2455
P95 pixel distance — maximum       : 8.0623
Maximum observed color distance    : 16.9115

Nearest-palette recovered levels:
Recovered minimum                  : 0
Recovered maximum                  : 253

Per-sample summary:
 sample_id  mean_nearest_rgb_distance  p95_nearest_rgb_distance  max_nearest_rgb_distance
       614                   3.693366                  8.062258                 16.911535
      1573                   3.019730                  8.062258                 16.911535
      1567                   2.366947                  7.810250                 16.911535
       642                   4.245487                  8.062258                 16.911535
      1596 

### **X03 STEP 3B — GRAD-CAM COLORMAP IDENTIFICATION AND RECOVERY BASIS LOCK**

The corrected color-distance inspection shows that the stored Grad-CAM PNG heatmaps are *very close to a JET-style palette*, but exact OpenCV JET encoding is not preserved.

This step compares the stored heatmap colors against the two relevant JET implementations—**OpenCV JET** and **Matplotlib `jet`**—and identifies the palette providing the lowest pixel-weighted RGB reconstruction error.

The selected palette will form the <u>locked scalar saliency recovery basis</u> for X03.

*No Grad-CAM–LIME spatial agreement metric is calculated in this step.*

In [22]:
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import cv2
import matplotlib

print("=" * 80)
print("X03 STEP 3B — GRAD-CAM COLORMAP IDENTIFICATION AND RECOVERY BASIS LOCK")
print("=" * 80)

X01 = Path("/kaggle/input/datasets/sazzad1/aqua20-x01-f06-final-gradcam-outputs")
g = pd.read_csv(X01 / "tables/X01_gradcam_metadata.csv")

levels = np.arange(256, dtype=np.uint8)

opencv_jet = cv2.applyColorMap(
    levels.reshape(256, 1),
    cv2.COLORMAP_JET
).reshape(256, 3)[:, ::-1].astype(np.float32)

mpl_jet = (
    matplotlib.colormaps["jet"](np.linspace(0, 1, 256))[:, :3] * 255
).astype(np.uint8).astype(np.float32)

palettes = {
    "opencv_jet": opencv_jet,
    "matplotlib_jet": mpl_jet,
}

records = []

for row in g.itertuples(index=False):
    heat = np.asarray(
        Image.open(X01 / row.heatmap_file).convert("RGB")
    )

    colors, counts = np.unique(
        heat.reshape(-1, 3),
        axis=0,
        return_counts=True
    )

    for name, palette in palettes.items():
        diff = colors[:, None, :].astype(np.float32) - palette[None, :, :]
        distances = np.sqrt(np.sum(diff ** 2, axis=2))
        nearest = distances.min(axis=1)

        records.append({
            "sample_id": row.sample_id,
            "palette": name,
            "weighted_mean_distance": np.average(nearest, weights=counts),
            "weighted_p95_distance": np.percentile(
                np.repeat(nearest, counts), 95
            ),
            "max_distance": nearest.max(),
        })

result = pd.DataFrame(records)

summary = result.groupby("palette").agg(
    mean_distance=("weighted_mean_distance", "mean"),
    max_sample_mean=("weighted_mean_distance", "max"),
    max_p95=("weighted_p95_distance", "max"),
    max_distance=("max_distance", "max"),
)

print("\nPalette comparison:")
print(summary.to_string())

winner_counts = (
    result.pivot(
        index="sample_id",
        columns="palette",
        values="weighted_mean_distance"
    )
    .idxmin(axis=1)
    .value_counts()
)

print("\nBest palette by sample:")
print(winner_counts.to_string())

selected_palette = summary["mean_distance"].idxmin()
all_samples_prefer_selected = winner_counts.get(selected_palette, 0) == 20

print("\n" + "-" * 80)
print(f"SELECTED RECOVERY PALETTE         : {selected_palette}")
print(f"Selected by all 20 samples        : {'PASS' if all_samples_prefer_selected else 'NO'}")
print("=" * 80)

assert all_samples_prefer_selected, (
    "A single common Grad-CAM palette could not be locked across all samples."
)

X03 STEP 3B — GRAD-CAM COLORMAP IDENTIFICATION AND RECOVERY BASIS LOCK

Palette comparison:
                mean_distance  max_sample_mean   max_p95  max_distance
palette                                                               
matplotlib_jet       0.000000         0.000000  0.000000      0.000000
opencv_jet           1.974435         4.245487  8.062258     16.911535

Best palette by sample:
matplotlib_jet    20

--------------------------------------------------------------------------------
SELECTED RECOVERY PALETTE         : matplotlib_jet
Selected by all 20 samples        : PASS


## **X03 STEP 4 — EXACT GRAD-CAM SCALAR SALIENCY RECOVERY AND INVERSE AMBIGUITY VERIFICATION**

Step 3B identified the stored Grad-CAM heatmaps as an **exact Matplotlib `jet` palette encoding** for all 20 frozen samples.

This step recovers normalized scalar Grad-CAM saliency from the saved RGB heatmaps and verifies the inverse mapping. Because quantized color maps may map more than one activation level to the same RGB value, any <u>palette-level ambiguity</u> is explicitly measured before the scalar representation is locked.

*No Grad-CAM–LIME spatial agreement or overlap metric is calculated in this step.*

In [23]:
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib

print("=" * 80)
print("X03 STEP 4 — EXACT GRAD-CAM SCALAR SALIENCY RECOVERY AND INVERSE AMBIGUITY VERIFICATION")
print("=" * 80)

X01 = Path("/kaggle/input/datasets/sazzad1/aqua20-x01-f06-final-gradcam-outputs")
g = pd.read_csv(X01 / "tables/X01_gradcam_metadata.csv")

palette = (
    matplotlib.colormaps["jet"](np.linspace(0, 1, 256))[:, :3] * 255
).astype(np.uint8)

color_to_levels = {}
for level, rgb in enumerate(palette):
    color_to_levels.setdefault(tuple(rgb), []).append(level)

ambiguous_colors = {
    rgb: levels
    for rgb, levels in color_to_levels.items()
    if len(levels) > 1
}

records = []

for row in g.itertuples(index=False):
    heat = np.asarray(
        Image.open(X01 / row.heatmap_file).convert("RGB")
    )

    flat = heat.reshape(-1, 3)
    rgb_keys = [tuple(rgb) for rgb in flat]

    recovered = np.array([
        np.mean(color_to_levels[rgb])
        for rgb in rgb_keys
    ], dtype=np.float32)

    ambiguous = np.array([
        len(color_to_levels[rgb]) > 1
        for rgb in rgb_keys
    ])

    recovered_map = recovered.reshape(heat.shape[:2]) / 255.0

    reconstructed_levels = np.rint(recovered).astype(np.int16)
    reconstructed = palette[reconstructed_levels].reshape(heat.shape)

    records.append({
        "sample_id": row.sample_id,
        "ambiguous_pixel_fraction": ambiguous.mean(),
        "recovered_min": recovered_map.min(),
        "recovered_max": recovered_map.max(),
        "recovered_mean": recovered_map.mean(),
        "exact_rgb_roundtrip": np.array_equal(heat, reconstructed),
    })

result = pd.DataFrame(records)

print(f"Palette levels                     : {len(palette)}")
print(f"Unique palette RGB colors          : {len(color_to_levels)}")
print(f"Ambiguous RGB colors               : {len(ambiguous_colors)}")

print("\nAmbiguous palette entries:")
for rgb, levels in ambiguous_colors.items():
    print(f"RGB {rgb} -> levels {levels}")

print("\nScalar recovery:")
print(f"Recovered saliency minimum         : {result['recovered_min'].min():.6f}")
print(f"Recovered saliency maximum         : {result['recovered_max'].max():.6f}")
print(f"Mean ambiguous pixel fraction      : {result['ambiguous_pixel_fraction'].mean():.6f}")
print(f"Maximum ambiguous pixel fraction   : {result['ambiguous_pixel_fraction'].max():.6f}")
print(f"Exact RGB round-trip samples       : {result['exact_rgb_roundtrip'].sum()}/20")

recovery_ready = (
    len(result) == 20
    and result["exact_rgb_roundtrip"].all()
)

print("-" * 80)
print(f"SCALAR SALIENCY RECOVERY VERIFIED  : {'PASS' if recovery_ready else 'FAIL'}")
print(f"GRAD-CAM RECOVERY BASIS LOCK READY : {'YES' if recovery_ready else 'NO'}")
print("=" * 80)

assert recovery_ready, "Grad-CAM scalar saliency recovery verification failed."

X03 STEP 4 — EXACT GRAD-CAM SCALAR SALIENCY RECOVERY AND INVERSE AMBIGUITY VERIFICATION
Palette levels                     : 256
Unique palette RGB colors          : 253
Ambiguous RGB colors               : 1

Ambiguous palette entries:
RGB (np.uint8(0), np.uint8(0), np.uint8(255)) -> levels [29, 30, 31, 32]

Scalar recovery:
Recovered saliency minimum         : 0.000000
Recovered saliency maximum         : 0.992157
Mean ambiguous pixel fraction      : 0.019005
Maximum ambiguous pixel fraction   : 0.045715
Exact RGB round-trip samples       : 20/20
--------------------------------------------------------------------------------
SCALAR SALIENCY RECOVERY VERIFIED  : PASS
GRAD-CAM RECOVERY BASIS LOCK READY : YES
